# Lab 1: Distributions and Bayes' Theorem

Probability becomes intuitive when you can sample from distributions and watch the theory materialize in histograms. This lab takes the core results from r1–r3 — PMFs, PDFs, joint distributions, the law of total probability, and Bayes' theorem — and makes them concrete in NumPy and SciPy.

**Sections**
1. PMFs and PDFs
2. CDFs and Interval Probabilities
3. Joint Distributions and Marginalization
4. Law of Total Probability (Simulation)
5. Bayes' Theorem: Disease Screening Sensitivity Sweep

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
rng = np.random.default_rng(42)
print('Setup complete.')

## 1. PMFs and PDFs

A PMF assigns probability to each discrete outcome; a PDF is a density whose integral gives probability.
A key point: **PDF values are not probabilities** — the PDF can exceed 1.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# Bernoulli PMF
theta = 0.3
axes[0].bar([0, 1], [1-theta, theta], color='steelblue', width=0.4)
axes[0].set_title(f'Bernoulli(θ={theta})')
axes[0].set_xlabel('x')
axes[0].set_ylabel('P(X=x)')
axes[0].set_xticks([0, 1])
print(f"Bernoulli PMF sums to: {(1-theta) + theta:.4f}")

# Poisson PMF
lam = 3.5
k = np.arange(0, 15)
pmf = stats.poisson.pmf(k, lam)
axes[1].bar(k, pmf, color='steelblue', width=0.7)
axes[1].set_title(f'Poisson(λ={lam})')
axes[1].set_xlabel('k')
print(f"Poisson PMF sums to: {pmf.sum():.6f}")

# Uniform PDF — can exceed 1!
x_unif = np.linspace(-0.1, 0.6, 300)
pdf_unif = stats.uniform(0, 0.1).pdf(x_unif)  # Uniform(0, 0.1) → density = 10
axes[2].plot(x_unif, pdf_unif, 'steelblue', lw=2)
axes[2].fill_between(x_unif, pdf_unif, alpha=0.3, color='steelblue')
axes[2].set_title('Uniform(0, 0.1) — density=10 > 1!')
axes[2].set_xlabel('x')
area = np.trapz(pdf_unif, x_unif)
print(f"Uniform(0,0.1) PDF integrates to: {area:.6f}")

# Gaussian PDF
x_norm = np.linspace(-4, 4, 300)
pdf_norm = stats.norm.pdf(x_norm)
axes[3].plot(x_norm, pdf_norm, 'steelblue', lw=2)
axes[3].fill_between(x_norm, pdf_norm, alpha=0.3, color='steelblue')
axes[3].set_title('N(0, 1)')
axes[3].set_xlabel('x')
area_norm = np.trapz(pdf_norm, x_norm)
print(f"N(0,1) PDF integrates to: {area_norm:.6f}")

plt.suptitle('PMFs (discrete) and PDFs (continuous)', y=1.02)
plt.tight_layout()
plt.show()

## 2. CDFs and Interval Probabilities

The CDF $F(x) = P(X \leq x)$. For any interval: $P(a < X \leq b) = F(b) - F(a)$.

In [ ]:
dist = stats.norm(0, 1)
x = np.linspace(-4, 4, 400)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(x, dist.pdf(x), 'steelblue', lw=2)
a, b = -1.0, 1.5
mask = (x >= a) & (x <= b)
ax1.fill_between(x, dist.pdf(x), where=mask, alpha=0.4, color='coral',
                 label=f'P({a} ≤ X ≤ {b})')
prob = dist.cdf(b) - dist.cdf(a)
ax1.set_title(f'P({a} ≤ X ≤ {b}) = F({b}) − F({a}) = {prob:.4f}')
ax1.set_xlabel('x')
ax1.legend()

ax2.plot(x, dist.cdf(x), 'steelblue', lw=2)
ax2.axvline(a, color='gray', linestyle=':', alpha=0.7)
ax2.axvline(b, color='gray', linestyle=':', alpha=0.7)
ax2.annotate('', xy=(b, dist.cdf(b)), xytext=(b, dist.cdf(a)),
             arrowprops=dict(arrowstyle='<->', color='coral', lw=2))
ax2.text(b + 0.1, (dist.cdf(a) + dist.cdf(b))/2, f'{prob:.3f}',
         color='coral', va='center')
ax2.set_title('CDF F(x) = P(X ≤ x)')
ax2.set_xlabel('x')

plt.tight_layout()
plt.show()

# Practical query: what's P(X > 2)?
p_tail = 1 - dist.cdf(2)
print(f"P(X > 2) = 1 - F(2) = {p_tail:.4f}  (~{p_tail*100:.2f}%)")

## 3. Joint Distributions and Marginalization

A joint distribution $p(x, y)$ encodes the full relationship between two variables. Marginalization — integrating out one variable — recovers the distribution of the other.

In [ ]:
# Sample from a 2D Gaussian with correlation
mu = [1.0, -0.5]
cov = [[1.0, 0.7],
       [0.7, 1.5]]
N = 3000
samples = rng.multivariate_normal(mu, cov, N)
x_samp, y_samp = samples[:, 0], samples[:, 1]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Joint scatter
axes[0].scatter(x_samp, y_samp, alpha=0.15, s=8, color='steelblue')
axes[0].set_xlabel('X')
axes[0].set_ylabel('Y')
axes[0].set_title('Joint distribution p(x, y)')

# Marginal of X (collapse y)
x_line = np.linspace(-4, 6, 300)
marginal_x = stats.norm(mu[0], np.sqrt(cov[0][0]))
axes[1].hist(x_samp, bins=50, density=True, color='steelblue', alpha=0.5, label='empirical')
axes[1].plot(x_line, marginal_x.pdf(x_line), 'coral', lw=2, label='theoretical N(1,1)')
axes[1].set_xlabel('X')
axes[1].set_title('Marginal p(x) — integrate out y')
axes[1].legend(fontsize=9)

# Marginal of Y
y_line = np.linspace(-5, 4, 300)
marginal_y = stats.norm(mu[1], np.sqrt(cov[1][1]))
axes[2].hist(y_samp, bins=50, density=True, color='steelblue', alpha=0.5, label='empirical')
axes[2].plot(y_line, marginal_y.pdf(y_line), 'coral', lw=2, label='theoretical N(-0.5, 1.5)')
axes[2].set_xlabel('Y')
axes[2].set_title('Marginal p(y) — integrate out x')
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.show()

## 4. Law of Total Probability (Simulation)

Quiz q1 asked: a binary classifier predicts class A 60% of the time (error rate 10%) and class B 40% of the time (error rate 30%). The overall error rate is:
$$P(\text{error}) = 0.10 \times 0.60 + 0.30 \times 0.40 = 0.18$$

Let's verify this by simulation.

In [ ]:
N = 500_000
# Step 1: assign class (A with prob 0.6, B with prob 0.4)
classes = rng.choice(['A', 'B'], size=N, p=[0.6, 0.4])

# Step 2: simulate errors conditional on class
errors = np.zeros(N, dtype=bool)
class_A = classes == 'A'
class_B = classes == 'B'
errors[class_A] = rng.random(class_A.sum()) < 0.10
errors[class_B] = rng.random(class_B.sum()) < 0.30

sim_error = errors.mean()
analytic_error = 0.10 * 0.60 + 0.30 * 0.40

print(f"Simulated overall error rate:  {sim_error:.5f}")
print(f"Analytic (law of total prob):  {analytic_error:.5f}")
print(f"Difference: {abs(sim_error - analytic_error):.5f}  (vanishes with more samples)")

## 5. Bayes' Theorem: Disease Screening Sensitivity Sweep

Quiz q1 showed that a test with 95% sensitivity and 10% false-positive rate gives only ~9% posterior probability when disease prevalence is 1%.

The **base-rate fallacy**: intuition overestimates the posterior because it ignores the prior. Let's sweep $P(D)$ from 0.1% to 50% and watch $P(D|+)$ climb.

In [ ]:
sensitivity = 0.95   # P(+ | disease)
fpr = 0.10           # P(+ | no disease)

priors = np.linspace(0.001, 0.50, 500)

def bayes_posterior(prior, sens, fpr):
    p_pos = sens * prior + fpr * (1 - prior)
    return (sens * prior) / p_pos

posteriors = bayes_posterior(priors, sensitivity, fpr)

plt.figure(figsize=(9, 5))
plt.plot(priors * 100, posteriors * 100, 'steelblue', lw=2.5)
plt.axhline(50, color='gray', linestyle=':', alpha=0.6, label='50% posterior')
# Mark the 1% prior from the quiz
p1_post = bayes_posterior(0.01, sensitivity, fpr)
plt.scatter([1], [p1_post*100], color='coral', s=100, zorder=5,
            label=f'Quiz example: P(D)=1% → P(D|+)≈{p1_post*100:.1f}%')
# Where does the posterior exceed 50%?
cross_50 = priors[posteriors >= 0.50][0]
plt.axvline(cross_50*100, color='forestgreen', linestyle='--', alpha=0.7,
            label=f'P(D|+)>50% when P(D)>{cross_50*100:.1f}%')
plt.xlabel('Disease prevalence P(D) (%)')
plt.ylabel('Posterior P(D | positive test) (%)')
plt.title('Bayes posterior vs prior — sensitivity=0.95, FPR=0.10')
plt.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f"At 1% prevalence:  P(D|+) = {p1_post*100:.1f}%  (most positives are false alarms)")
print(f"Test is >50% reliable only when prevalence > {cross_50*100:.1f}%")